# A2.1 · Agent identity: user, workload, agent

**Function A — Securing AI Architectures → Securing the Architecture — Identity and Ingress**  ·  *Security of AI*

Builds on **[A1.17 · The CyberTravels risk register](https://spbreed.github.io/cyber-commons/lessons/A1.17.html)**.

| | |
|---|---|
| Tools used | SPIFFE/SPIRE, Keycloak |

> **Runs anywhere.** Every line of code is in this notebook — nothing to install, nothing to clone, no API key, no network. Standard library only, so it works on a Kaggle kernel with the internet switched off — and where a lesson involves a model, the same code calls an open-weight endpoint or a frontier API when you configure one.

## 1 · The hook

Three identities are present every time an agent acts: the person who asked, the workload that runs, and the agent instance doing the work. Collapse any two of them and you lose the ability to answer the only question that matters after an incident.

> **At CyberTravels.** Three identities are present whenever CyberTravels books a flight: the traveller who asked, the workload the agent runs as, and which of the four agents is acting. CyberTravels collapses all three into `cybertravels-svc`, which is R11.

## 2 · The framework

```
   who asked        what runs         what acted
   +----------+     +-----------+     +--------------+
   |  human   | --> | workload  | --> | agent        |
   | dana@..  |     | pod/task  |     | instance #7  |
   +----------+     +-----------+     +--------------+
        |                |                   |
      consent         attestation        the actor in the log

   collapse any two and the post-incident question loses its answer
```

**Mitigates: T9 Identity Spoofing · T3 Privilege Compromise · T1 Memory Poisoning.**

Three identities are in play whenever an agent acts, and A1.6 and A1.7 were both
caused by collapsing them into one.

**The user.** The human who asked. Carries the business authority and is the
answer to "on whose behalf".

**The workload.** The process that runs — a container, a function, a pod. It has
its own identity, derived from the platform, not from a secret someone pasted.

**The agent instance.** This particular run, of this particular agent, for this
particular task. It is what you revoke when one agent misbehaves.

The control is to keep all three, and to use them for different things:

- **Authorize on the workload.** What may this agent ever do? That is its
  ceiling, and it does not change per request.
- **Attribute to the user.** Who caused this? That is what the audit trail needs
  and what A1.13 could not answer.
- **Scope memory and state to the instance or the user**, never to the workload
  alone — which is the write that made A1.4 spread across sessions.

The failure mode to watch for is a system that authenticates the agent and then
forgets the human, because it produces logs that are complete and useless.

> **What this control closes.**
>
> Answers **who is calling**, so every later control has a subject. Without it, default-deny has nothing to deny and the audit trail has nobody to name.

## 3 · The control

In [ ]:
from dataclasses import dataclass, field

@dataclass(frozen=True)
class Principal:
    user: str            # the human who asked
    workload: str        # the process, from platform attestation
    instance: str        # this run of this agent
    scopes: frozenset    # the workload's ceiling

def authorize(p, required):
    """Authorization is about the WORKLOAD: what may this agent ever do."""
    return required in p.scopes

def attribute(p, action):
    """Attribution is about the USER: who caused this."""
    return {"action": action, "caused_by": p.user,
            "performed_by": p.workload, "run": p.instance}

def memory_key(p, workspace):
    """Memory is scoped to the USER, not the workspace - this is the write that
    let A1.4 leak a poisoned note between people."""
    return f"{workspace}:{p.user}"

dana = Principal("dana@corp", "reports-agent", "run-8812",
                 frozenset({"reports:read"}))
priya = Principal("priya@corp", "reports-agent", "run-8813",
                  frozenset({"reports:read"}))

print(f"{'request':28s}{'authorized?':13s}attributed to")
for p, need in ((dana, "reports:read"), (dana, "db:admin")):
    ok = authorize(p, need)
    rec = attribute(p, need)
    print(f"{p.user + ' -> ' + need:28s}{str(ok):13s}{rec['caused_by']} via {rec['performed_by']}")

print("\nmemory keys - the same workspace, two users:")
print(f"   dana  -> {memory_key(dana, 'acme')}")
print(f"   priya -> {memory_key(priya, 'acme')}")
print(f"   shared? {memory_key(dana, 'acme') == memory_key(priya, 'acme')}")
print()
print("db:admin is refused because the WORKLOAD never held it - so no user can")
print("borrow it through the agent, which is A1.6 closed. The audit line names")
print("dana, which is A1.13 closed. And a note written in dana's session cannot")
print("be read back in priya's, which is A1.4 closed.")
assert not authorize(dana, "db:admin")
assert memory_key(dana, "acme") != memory_key(priya, "acme")

## What you just proved

Authorization resolves against the workload ceiling and refuses `db:admin` no matter who asks, attribution names the human on every action, and memory keys differ per user so a note written in one session cannot be read back in another's.

## Your turn

For one agent, write down its three identities. If the workload and the user are the same value, you have inherited credentials; if the instance does not exist, you cannot revoke one run.

---

**Next → [A2.2 · Bootstrapping the first credential](https://spbreed.github.io/cyber-commons/lessons/A2.2.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/A2.1.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/A2.1.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*